# Latest results: MAE, mean overwrite time and the 90% threshold

**On prn_1, MAE is below even the lower bound on mean time to overwrite, but the requirement that `error < overwrite` for at least 90% of records is not met.** These are two different criteria.

Predictors are selected by minimum validation MAE on day 4. For prn_1, the selection is a **25% log-L2 + 75% raw-L1** mixture in seconds; for proj_1, it is the per-address median. Days 5–7 have already been inspected: this is development evaluation, not a new holdout.

- `same_start`: the next Read/Write with the same starting Offset, regardless of size.
- `range`: the next Read/Write whose byte range intersects the written range; partial overlap is sufficient, not a complete overwrite.

## Main comparison: all writes with an observed Read, days 5–7

MAE and mean overwrite time refer to the same records. Unknown overwrite times are retained: the observation-window boundary gives only a lower bound on the interval, not its actual value. If MAE is below this lower bound on the mean, the condition is proven.

The 90% criterion is `P(abs(prediction − target) < overwrite) ≥ 0.90`. The lower bound counts confirmed passes; the upper bound additionally treats every unresolved case as a pass. If even the upper bound is below 90%, the condition definitely fails.

| Workload / rule | Method | Observed Read N / all Write N | MAE, s | Mean overwrite ≥, s | MAE / lower bound | MAE < mean | Fraction error < overwrite | ≥90% |
|---|---|---:|---:|---:|---:|---|---:|---|
| proj_1 / same_start | address_median | 211,844 / 1,022,738 | 18162.30 | 18974.56 | 0.9572 | proven | 51.35%–52.72% | no |
| proj_1 / range | address_median | 236,034 / 1,022,738 | 17146.95 | 16275.23 | 1.0536 | undetermined | 45.79%–47.17% | no |
| prn_1 / same_start | mix_log25 | 633,316 / 1,299,820 | 13715.20 | 19184.55 | 0.7149 | proven | 50.03%–50.36% | no |
| prn_1 / range | mix_log25 | 684,054 / 1,299,820 | 13741.74 | 16475.83 | 0.8341 | proven | 46.19%–46.51% | no |

For prn_1, the mean-time condition also holds on each of days 5, 6 and 7 separately. A condition on averages does not guarantee small errors for most records. MAE is undefined for writes without an observed Read, so this result does not extend to all writes.

## Control comparison: observed Read and known overwrite only

This is a different cohort. Here every evaluated candidate has MAE above mean overwrite time. Unknown intervals are excluded, so the conclusion differs from the main table.

| Workload / rule | Paired N | MAE, s | Mean overwrite, s | MAE / mean | MAE < mean |
|---|---:|---:|---:|---:|---|
| proj_1 / same_start | 188,375 | 17877.65 | 12825.59 | 1.3939 | no |
| proj_1 / range | 212,376 | 17074.68 | 10522.68 | 1.6227 | no |
| prn_1 / same_start | 522,581 | 12471.49 | 9183.09 | 1.3581 | no |
| prn_1 / range | 591,037 | 12551.49 | 8571.29 | 1.4644 | no |

## Evaluated candidates and the 90% threshold

| Workload / rule | Candidates | Highest upper bound on pass fraction | ≥90% reached |
|---|---:|---:|---|
| proj_1 / same_start | 5 | 58.76% | no |
| proj_1 / range | 5 | 58.64% | no |
| prn_1 / same_start | 8 | 55.48% | no |
| prn_1 / range | 8 | 50.76% | no |

Numerical results, daily metrics and source fingerprints are embedded in results.ipynb and saved in overwrite-results.json. Mean-overwrite calculations were independently verified for 20 windows; the parent long-history/mixture metrics were also verified.

Source results in the research repository: results/mean-overwrite/, results/long-history/{proj_1,prn_1}/, results/mixture/prn_1/. Protocol: docs/protocols/mean-overwrite-protocol.md.


In [1]:
overwrite_results = {
    'evaluation_role': 'development; days 5–7 already inspected',
    'selection': 'minimum observed-Read MAE on validation day 4; no pass-count constraint',
    'ninety_percent_threshold': 0.9,
    'selected_results': [
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'validation',
            'method': 'address_median',
            'observed_reads': 111_801,
            'all_writes': 359_386,
            'paired_n': 105_507,
            'censored_n': 6_294,
            'observed_read': {
                'mae_seconds': 11082.821313910727,
                'mean_overwrite_lower_seconds': 19501.20653418926,
                'ratio_upper_bound': 0.5683146473260754,
                'margin_lower_seconds': 8418.385220278533,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 10906.47195984205,
                'mean_overwrite_seconds': 16955.60452535874,
                'ratio': 0.6432369865391925,
                'margin_seconds': 6049.132565516689,
                'passed': True,
            },
            'confirmed_pass_count': 72_020,
            'unknown_pass_count': 745,
            'pass_lower': 0.6441802846128389,
            'pass_upper': 0.6508439101618053,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'day5',
            'method': 'address_median',
            'observed_reads': 24_707,
            'all_writes': 234_914,
            'paired_n': 17_999,
            'censored_n': 6_708,
            'observed_read': {
                'mae_seconds': 20505.715189505,
                'mean_overwrite_lower_seconds': 26151.59791592412,
                'ratio_upper_bound': 0.7841094550103475,
                'margin_lower_seconds': 5645.88272641912,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 21055.379672835945,
                'mean_overwrite_seconds': 9474.905258464027,
                'ratio': 2.2222258796758907,
                'margin_seconds': -11580.474414371918,
                'passed': False,
            },
            'confirmed_pass_count': 9_423,
            'unknown_pass_count': 893,
            'pass_lower': 0.3813898895049986,
            'pass_upper': 0.4175334925324807,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'day6',
            'method': 'address_median',
            'observed_reads': 66_311,
            'all_writes': 314_624,
            'paired_n': 56_276,
            'censored_n': 10_035,
            'observed_read': {
                'mae_seconds': 33227.52535223575,
                'mean_overwrite_lower_seconds': 16777.197536403713,
                'ratio_upper_bound': 1.9805170249761666,
                'margin_lower_seconds': -16450.32781583204,
                'status': 'undetermined',
            },
            'paired': {
                'mae_seconds': 34614.64146376193,
                'mean_overwrite_seconds': 6709.64116966601,
                'ratio': 5.158940782146919,
                'margin_seconds': -27905.00029409592,
                'passed': False,
            },
            'confirmed_pass_count': 16_581,
            'unknown_pass_count': 909,
            'pass_lower': 0.25004901147622566,
            'pass_upper': 0.26375714436518827,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'day7',
            'method': 'address_median',
            'observed_reads': 120_826,
            'all_writes': 473_200,
            'paired_n': 114_100,
            'censored_n': 6_726,
            'observed_read': {
                'mae_seconds': 9415.102926920612,
                'mean_overwrite_lower_seconds': 18712.90732208477,
                'ratio_upper_bound': 0.5031341610829766,
                'margin_lower_seconds': 9297.80439516416,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 9121.405143138176,
                'mean_overwrite_seconds': 16370.64441546479,
                'ratio': 0.5571805795574849,
                'margin_seconds': 7249.239272326615,
                'passed': True,
            },
            'confirmed_pass_count': 82_788,
            'unknown_pass_count': 1_098,
            'pass_lower': 0.6851836525251188,
            'pass_upper': 0.6942711005909324,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'address_median',
            'observed_reads': 211_844,
            'all_writes': 1_022_738,
            'paired_n': 188_375,
            'censored_n': 23_469,
            'observed_read': {
                'mae_seconds': 18162.300395891856,
                'mean_overwrite_lower_seconds': 18974.556823150142,
                'ratio_upper_bound': 0.9571923373584524,
                'margin_lower_seconds': 812.2564272582858,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 17877.64654852346,
                'mean_overwrite_seconds': 12825.593173275387,
                'ratio': 1.3939040718814475,
                'margin_seconds': -5052.0533752480715,
                'passed': False,
            },
            'confirmed_pass_count': 108_792,
            'unknown_pass_count': 2_900,
            'pass_lower': 0.5135477049149374,
            'pass_upper': 0.5272370234701006,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'validation',
            'method': 'address_median',
            'observed_reads': 119_387,
            'all_writes': 359_386,
            'paired_n': 113_732,
            'censored_n': 5_655,
            'observed_read': {
                'mae_seconds': 11018.689681322365,
                'mean_overwrite_lower_seconds': 17527.782956169318,
                'ratio_upper_bound': 0.6286413808794954,
                'margin_lower_seconds': 6509.093274846953,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 10931.105762710868,
                'mean_overwrite_seconds': 15116.984711879884,
                'ratio': 0.7231009338866706,
                'margin_seconds': 4185.878949169017,
                'passed': True,
            },
            'confirmed_pass_count': 71_320,
            'unknown_pass_count': 705,
            'pass_lower': 0.5973849749135165,
            'pass_upper': 0.6032901404675551,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'day5',
            'method': 'address_median',
            'observed_reads': 32_035,
            'all_writes': 234_914,
            'paired_n': 23_785,
            'censored_n': 8_250,
            'observed_read': {
                'mae_seconds': 18079.90414571119,
                'mean_overwrite_lower_seconds': 23862.0210781754,
                'ratio_upper_bound': 0.7576853648095789,
                'margin_lower_seconds': 5782.116932464211,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 18724.477170511094,
                'mean_overwrite_seconds': 7211.579272932698,
                'ratio': 2.5964461405548,
                'margin_seconds': -11512.897897578398,
                'passed': False,
            },
            'confirmed_pass_count': 11_120,
            'unknown_pass_count': 915,
            'pass_lower': 0.34712033713126267,
            'pass_upper': 0.3756828468862182,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'day6',
            'method': 'address_median',
            'observed_reads': 74_854,
            'all_writes': 314_624,
            'paired_n': 65_434,
            'censored_n': 9_420,
            'observed_read': {
                'mae_seconds': 29778.947843886162,
                'mean_overwrite_lower_seconds': 14228.296806627495,
                'ratio_upper_bound': 2.092938336092007,
                'margin_lower_seconds': -15550.651037258667,
                'status': 'undetermined',
            },
            'paired': {
                'mae_seconds': 30942.828607788506,
                'mean_overwrite_seconds': 5987.514879484697,
                'ratio': 5.167891726467247,
                'margin_seconds': -24955.31372830381,
                'passed': False,
            },
            'confirmed_pass_count': 16_356,
            'unknown_pass_count': 1_115,
            'pass_lower': 0.2185053570951452,
            'pass_upper': 0.23340102065353888,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'day7',
            'method': 'address_median',
            'observed_reads': 129_145,
            'all_writes': 473_200,
            'paired_n': 123_157,
            'censored_n': 5_988,
            'observed_read': {
                'mae_seconds': 9593.859921873815,
                'mean_overwrite_lower_seconds': 15579.715687676697,
                'ratio_upper_bound': 0.6157917200929669,
                'margin_lower_seconds': 5985.855765802882,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 9387.830329441505,
                'mean_overwrite_seconds': 13571.701985795014,
                'ratio': 0.6917209307474768,
                'margin_seconds': 4183.871656353509,
                'passed': True,
            },
            'confirmed_pass_count': 80_598,
            'unknown_pass_count': 1_239,
            'pass_lower': 0.6240892020597003,
            'pass_upper': 0.6336830694180959,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'address_median',
            'observed_reads': 236_034,
            'all_writes': 1_022_738,
            'paired_n': 212_376,
            'censored_n': 23_658,
            'observed_read': {
                'mae_seconds': 17146.94548592367,
                'mean_overwrite_lower_seconds': 16275.227962444607,
                'ratio_upper_bound': 1.0535610023706314,
                'margin_lower_seconds': -871.7175234790611,
                'status': 'undetermined',
            },
            'paired': {
                'mae_seconds': 17074.67772491085,
                'mean_overwrite_seconds': 10522.679413377513,
                'ratio': 1.622654939311727,
                'margin_seconds': -6551.998311533336,
                'passed': False,
            },
            'confirmed_pass_count': 108_074,
            'unknown_pass_count': 3_269,
            'pass_lower': 0.4578747129650813,
            'pass_upper': 0.47172441258462766,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'validation',
            'method': 'mix_log25',
            'observed_reads': 157_855,
            'all_writes': 345_688,
            'paired_n': 135_545,
            'censored_n': 22_310,
            'observed_read': {
                'mae_seconds': 12644.232627937901,
                'mean_overwrite_lower_seconds': 16488.486690531226,
                'ratio_upper_bound': 0.7668522202949681,
                'margin_lower_seconds': 3844.254062593325,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12670.605091518037,
                'mean_overwrite_seconds': 8189.394985479036,
                'ratio': 1.547196723809857,
                'margin_seconds': -4481.2101060390005,
                'passed': False,
            },
            'confirmed_pass_count': 69_588,
            'unknown_pass_count': 695,
            'pass_lower': 0.44083494346077096,
            'pass_upper': 0.44523771815907004,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'day5',
            'method': 'mix_log25',
            'observed_reads': 251_222,
            'all_writes': 487_596,
            'paired_n': 209_721,
            'censored_n': 41_501,
            'observed_read': {
                'mae_seconds': 13362.356321008185,
                'mean_overwrite_lower_seconds': 20073.929287856554,
                'ratio_upper_bound': 0.6656572377731528,
                'margin_lower_seconds': 6711.572966848369,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12966.911817795848,
                'mean_overwrite_seconds': 11206.52402773403,
                'ratio': 1.1570859782841842,
                'margin_seconds': -1760.3877900618172,
                'passed': False,
            },
            'confirmed_pass_count': 125_809,
            'unknown_pass_count': 1_581,
            'pass_lower': 0.5007881475348497,
            'pass_upper': 0.507081386184331,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'day6',
            'method': 'mix_log25',
            'observed_reads': 197_159,
            'all_writes': 410_944,
            'paired_n': 161_174,
            'censored_n': 35_985,
            'observed_read': {
                'mae_seconds': 12529.233661693539,
                'mean_overwrite_lower_seconds': 17898.396667134493,
                'ratio_upper_bound': 0.7000198897535915,
                'margin_lower_seconds': 5369.163005440954,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 10913.896512435744,
                'mean_overwrite_seconds': 6990.429909595459,
                'ratio': 1.5612625623289222,
                'margin_seconds': -3923.4666028402844,
                'passed': False,
            },
            'confirmed_pass_count': 99_860,
            'unknown_pass_count': 236,
            'pass_lower': 0.5064947580379288,
            'pass_upper': 0.5076917614717056,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'day7',
            'method': 'mix_log25',
            'observed_reads': 184_935,
            'all_writes': 401_280,
            'paired_n': 151_686,
            'censored_n': 33_249,
            'observed_read': {
                'mae_seconds': 15458.857281303895,
                'mean_overwrite_lower_seconds': 19347.56549613187,
                'ratio_upper_bound': 0.7990078795388785,
                'margin_lower_seconds': 3888.7082148279733,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 13441.530990872132,
                'mean_overwrite_seconds': 8715.30793336798,
                'ratio': 1.5422898529389921,
                'margin_seconds': -4726.223057504152,
                'passed': False,
            },
            'confirmed_pass_count': 91_201,
            'unknown_pass_count': 266,
            'pass_lower': 0.49315164787628085,
            'pass_upper': 0.4945899910779463,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'mix_log25',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 13715.195621932955,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.7149082531530696,
                'margin_lower_seconds': 5469.357866494018,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12471.486984250692,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.358092412667564,
                'margin_seconds': -3288.395415574443,
                'passed': False,
            },
            'confirmed_pass_count': 316_870,
            'unknown_pass_count': 2_083,
            'pass_lower': 0.5003347460035749,
            'pass_upper': 0.5036237833877559,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'validation',
            'method': 'mix_log25',
            'observed_reads': 173_485,
            'all_writes': 345_688,
            'paired_n': 155_726,
            'censored_n': 17_759,
            'observed_read': {
                'mae_seconds': 11695.650309638444,
                'mean_overwrite_lower_seconds': 14279.253252911953,
                'ratio_upper_bound': 0.8190659625182685,
                'margin_lower_seconds': 2583.6029432735086,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 11509.372590508836,
                'mean_overwrite_seconds': 8025.54628373885,
                'ratio': 1.4340921083252391,
                'margin_seconds': -3483.8263067699854,
                'passed': False,
            },
            'confirmed_pass_count': 77_881,
            'unknown_pass_count': 573,
            'pass_lower': 0.4489206559644926,
            'pass_upper': 0.4522235351759518,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'day5',
            'method': 'mix_log25',
            'observed_reads': 266_018,
            'all_writes': 487_596,
            'paired_n': 233_049,
            'censored_n': 32_969,
            'observed_read': {
                'mae_seconds': 13678.476194532093,
                'mean_overwrite_lower_seconds': 17293.43267041247,
                'ratio_upper_bound': 0.7909636250491063,
                'margin_lower_seconds': 3614.9564758803754,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 13225.788481783562,
                'mean_overwrite_seconds': 10548.082455434673,
                'ratio': 1.2538571382677484,
                'margin_seconds': -2677.7060263488893,
                'passed': False,
            },
            'confirmed_pass_count': 123_821,
            'unknown_pass_count': 1_444,
            'pass_lower': 0.46546098384319856,
            'pass_upper': 0.4708891879496876,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'day6',
            'method': 'mix_log25',
            'observed_reads': 214_896,
            'all_writes': 410_944,
            'paired_n': 183_809,
            'censored_n': 31_087,
            'observed_read': {
                'mae_seconds': 12910.436842403611,
                'mean_overwrite_lower_seconds': 15303.389097714897,
                'ratio_upper_bound': 0.8436325287142701,
                'margin_lower_seconds': 2392.9522553112856,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 11412.3302849754,
                'mean_overwrite_seconds': 6534.244394447167,
                'ratio': 1.7465416957274593,
                'margin_seconds': -4878.085890528233,
                'passed': False,
            },
            'confirmed_pass_count': 98_849,
            'unknown_pass_count': 368,
            'pass_lower': 0.4599852952125679,
            'pass_upper': 0.46169775147047876,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'day7',
            'method': 'mix_log25',
            'observed_reads': 203_140,
            'all_writes': 401_280,
            'paired_n': 174_179,
            'censored_n': 28_961,
            'observed_read': {
                'mae_seconds': 14703.989559763151,
                'mean_overwrite_lower_seconds': 16645.433335679987,
                'ratio_upper_bound': 0.8833647801913758,
                'margin_lower_seconds': 1941.4437759168359,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12851.437751560641,
                'mean_overwrite_seconds': 8076.049395487098,
                'ratio': 1.5913025196133685,
                'margin_seconds': -4775.388356073543,
                'passed': False,
            },
            'confirmed_pass_count': 93_285,
            'unknown_pass_count': 357,
            'pass_lower': 0.45921531948409966,
            'pass_upper': 0.4609727281677661,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'mix_log25',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 13741.737575063507,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.8340545367993227,
                'margin_lower_seconds': 2734.088607475238,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12551.49233190359,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.4643639614877189,
                'margin_seconds': -3980.199496240033,
                'passed': False,
            },
            'confirmed_pass_count': 315_955,
            'unknown_pass_count': 2_169,
            'pass_lower': 0.46188604993173055,
            'pass_upper': 0.4650568522368117,
            'ninety_percent_status': 'failed',
        },
    ],
    'pooled_candidates': [
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'log_l2',
            'observed_reads': 211_844,
            'all_writes': 1_022_738,
            'paired_n': 188_375,
            'censored_n': 23_469,
            'observed_read': {
                'mae_seconds': 14481.23229421975,
                'mean_overwrite_lower_seconds': 18974.556823150142,
                'ratio_upper_bound': 0.7631921224400743,
                'margin_lower_seconds': 4493.324528930392,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 15240.914170314558,
                'mean_overwrite_seconds': 12825.593173275387,
                'ratio': 1.188320412507077,
                'margin_seconds': -2415.320997039171,
                'passed': False,
            },
            'confirmed_pass_count': 123_826,
            'unknown_pass_count': 121,
            'pass_lower': 0.5845150204867733,
            'pass_upper': 0.5850861955023507,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'raw_l1',
            'observed_reads': 211_844,
            'all_writes': 1_022_738,
            'paired_n': 188_375,
            'censored_n': 23_469,
            'observed_read': {
                'mae_seconds': 16095.13346006491,
                'mean_overwrite_lower_seconds': 18974.556823150142,
                'ratio_upper_bound': 0.8482481888814312,
                'margin_lower_seconds': 2879.4233630852323,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 16890.486906677535,
                'mean_overwrite_seconds': 12825.593173275387,
                'ratio': 1.3169361197166416,
                'margin_seconds': -4064.8937334021484,
                'passed': False,
            },
            'confirmed_pass_count': 124_162,
            'unknown_pass_count': 323,
            'pass_lower': 0.5861010932573025,
            'pass_upper': 0.5876258001170672,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'address_median',
            'observed_reads': 211_844,
            'all_writes': 1_022_738,
            'paired_n': 188_375,
            'censored_n': 23_469,
            'observed_read': {
                'mae_seconds': 18162.300395891856,
                'mean_overwrite_lower_seconds': 18974.556823150142,
                'ratio_upper_bound': 0.9571923373584524,
                'margin_lower_seconds': 812.2564272582858,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 17877.64654852346,
                'mean_overwrite_seconds': 12825.593173275387,
                'ratio': 1.3939040718814475,
                'margin_seconds': -5052.0533752480715,
                'passed': False,
            },
            'confirmed_pass_count': 108_792,
            'unknown_pass_count': 2_900,
            'pass_lower': 0.5135477049149374,
            'pass_upper': 0.5272370234701006,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'previous_model',
            'observed_reads': 211_844,
            'all_writes': 1_022_738,
            'paired_n': 188_375,
            'censored_n': 23_469,
            'observed_read': {
                'mae_seconds': 13771.977502162834,
                'mean_overwrite_lower_seconds': 18974.556823150142,
                'ratio_upper_bound': 0.7258128677535257,
                'margin_lower_seconds': 5202.579320987308,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 14249.228601673538,
                'mean_overwrite_seconds': 12825.593173275387,
                'ratio': 1.1109995778881068,
                'margin_seconds': -1423.635428398151,
                'passed': False,
            },
            'confirmed_pass_count': 112_585,
            'unknown_pass_count': 116,
            'pass_lower': 0.5314523894941561,
            'pass_upper': 0.5319999622363626,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'previous_address_median',
            'observed_reads': 211_844,
            'all_writes': 1_022_738,
            'paired_n': 188_375,
            'censored_n': 23_469,
            'observed_read': {
                'mae_seconds': 13097.407643726501,
                'mean_overwrite_lower_seconds': 18974.556823150142,
                'ratio_upper_bound': 0.690261583751292,
                'margin_lower_seconds': 5877.149179423641,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 13075.554209607675,
                'mean_overwrite_seconds': 12825.593173275387,
                'ratio': 1.0194892378820444,
                'margin_seconds': -249.96103633228813,
                'passed': False,
            },
            'confirmed_pass_count': 109_669,
            'unknown_pass_count': 189,
            'pass_lower': 0.5176875436642058,
            'pass_upper': 0.5185797095976284,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'log_l2',
            'observed_reads': 236_034,
            'all_writes': 1_022_738,
            'paired_n': 212_376,
            'censored_n': 23_658,
            'observed_read': {
                'mae_seconds': 12733.140969599392,
                'mean_overwrite_lower_seconds': 16275.227962444607,
                'ratio_upper_bound': 0.7823632946328833,
                'margin_lower_seconds': 3542.0869928452157,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 13541.080443576428,
                'mean_overwrite_seconds': 10522.679413377513,
                'ratio': 1.286847190874371,
                'margin_seconds': -3018.4010301989147,
                'passed': False,
            },
            'confirmed_pass_count': 138_133,
            'unknown_pass_count': 271,
            'pass_lower': 0.5852250099561928,
            'pass_upper': 0.5863731496309854,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'raw_l1',
            'observed_reads': 236_034,
            'all_writes': 1_022_738,
            'paired_n': 212_376,
            'censored_n': 23_658,
            'observed_read': {
                'mae_seconds': 13462.049839254494,
                'mean_overwrite_lower_seconds': 16275.227962444607,
                'ratio_upper_bound': 0.8271496946352104,
                'margin_lower_seconds': 2813.178123190113,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 14223.526570084818,
                'mean_overwrite_seconds': 10522.679413377513,
                'ratio': 1.3517019773502181,
                'margin_seconds': -3700.8471567073047,
                'passed': False,
            },
            'confirmed_pass_count': 134_232,
            'unknown_pass_count': 353,
            'pass_lower': 0.5686977299880526,
            'pass_upper': 0.5701932772397197,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'address_median',
            'observed_reads': 236_034,
            'all_writes': 1_022_738,
            'paired_n': 212_376,
            'censored_n': 23_658,
            'observed_read': {
                'mae_seconds': 17146.94548592367,
                'mean_overwrite_lower_seconds': 16275.227962444607,
                'ratio_upper_bound': 1.0535610023706314,
                'margin_lower_seconds': -871.7175234790611,
                'status': 'undetermined',
            },
            'paired': {
                'mae_seconds': 17074.67772491085,
                'mean_overwrite_seconds': 10522.679413377513,
                'ratio': 1.622654939311727,
                'margin_seconds': -6551.998311533336,
                'passed': False,
            },
            'confirmed_pass_count': 108_074,
            'unknown_pass_count': 3_269,
            'pass_lower': 0.4578747129650813,
            'pass_upper': 0.47172441258462766,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'previous_model',
            'observed_reads': 236_034,
            'all_writes': 1_022_738,
            'paired_n': 212_376,
            'censored_n': 23_658,
            'observed_read': {
                'mae_seconds': 12258.641352677616,
                'mean_overwrite_lower_seconds': 16275.227962444607,
                'ratio_upper_bound': 0.7532085806088038,
                'margin_lower_seconds': 4016.5866097669914,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12837.449819718071,
                'mean_overwrite_seconds': 10522.679413377513,
                'ratio': 1.219979181670952,
                'margin_seconds': -2314.770406340558,
                'passed': False,
            },
            'confirmed_pass_count': 114_664,
            'unknown_pass_count': 252,
            'pass_lower': 0.48579441944804563,
            'pass_upper': 0.48686206224526973,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'proj_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'previous_address_median',
            'observed_reads': 236_034,
            'all_writes': 1_022_738,
            'paired_n': 212_376,
            'censored_n': 23_658,
            'observed_read': {
                'mae_seconds': 12208.42282906123,
                'mean_overwrite_lower_seconds': 16275.227962444607,
                'ratio_upper_bound': 0.7501230002573478,
                'margin_lower_seconds': 4066.8051333833773,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12346.060941922848,
                'mean_overwrite_seconds': 10522.679413377513,
                'ratio': 1.1732811061626818,
                'margin_seconds': -1823.3815285453347,
                'passed': False,
            },
            'confirmed_pass_count': 109_922,
            'unknown_pass_count': 335,
            'pass_lower': 0.46570409347805825,
            'pass_upper': 0.46712338052992364,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'log_l2',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 15536.40506399272,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.8098392841597806,
                'margin_lower_seconds': 3648.148424434252,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12906.605080646535,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.4054749409960456,
                'margin_seconds': -3723.5135119702863,
                'passed': False,
            },
            'confirmed_pass_count': 349_987,
            'unknown_pass_count': 1_363,
            'pass_lower': 0.5526261771374795,
            'pass_upper': 0.5547783413019725,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'raw_l1',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 13545.950922374297,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.706086327760813,
                'margin_lower_seconds': 5638.602566052676,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12726.875633998548,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.385903161132601,
                'margin_seconds': -3543.7840653222993,
                'passed': False,
            },
            'confirmed_pass_count': 312_552,
            'unknown_pass_count': 2_627,
            'pass_lower': 0.4935166646666119,
            'pass_upper': 0.4976646729278907,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'address_median',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 13973.124539562252,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.728352867216404,
                'margin_lower_seconds': 5211.428948864721,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 14484.553707811696,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.5773069014381675,
                'margin_seconds': -5301.462139135447,
                'passed': False,
            },
            'confirmed_pass_count': 309_367,
            'unknown_pass_count': 6_678,
            'pass_lower': 0.4884875796600749,
            'pass_upper': 0.4990320787726822,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'previous_model',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 18796.79136905903,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.9797877954469015,
                'margin_lower_seconds': 387.76211936794425,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 15531.107575619215,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.6912722103954843,
                'margin_seconds': -6348.016006942966,
                'passed': False,
            },
            'confirmed_pass_count': 348_068,
            'unknown_pass_count': 11,
            'pass_lower': 0.5495960942088941,
            'pass_upper': 0.549613463105306,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'previous_address_median',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 14051.59650591611,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.7324432395256265,
                'margin_lower_seconds': 5132.956982510863,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12433.508292020631,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.3539566930195528,
                'margin_seconds': -3250.4167233443823,
                'passed': False,
            },
            'confirmed_pass_count': 315_888,
            'unknown_pass_count': 545,
            'pass_lower': 0.49878417725116686,
            'pass_upper': 0.49964472711884744,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'mix_log75',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 14648.584833499795,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.7635614163413557,
                'margin_lower_seconds': 4535.968654927177,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12480.968373328507,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.359124896010119,
                'margin_seconds': -3297.876804652258,
                'passed': False,
            },
            'confirmed_pass_count': 329_563,
            'unknown_pass_count': 1_642,
            'pass_lower': 0.5203768734723266,
            'pass_upper': 0.5229695760094487,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'mix_log50',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 14081.589047532121,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.7340066088077994,
                'margin_lower_seconds': 5102.964440894852,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12378.78556055525,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.3479976180113016,
                'margin_seconds': -3195.6939918790013,
                'passed': False,
            },
            'confirmed_pass_count': 322_370,
            'unknown_pass_count': 1_864,
            'pass_lower': 0.5090191942095257,
            'pass_upper': 0.5119624326560517,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'same_start',
            'window': 'pooled',
            'method': 'mix_log25',
            'observed_reads': 633_316,
            'all_writes': 1_299_820,
            'paired_n': 522_581,
            'censored_n': 110_735,
            'observed_read': {
                'mae_seconds': 13715.195621932955,
                'mean_overwrite_lower_seconds': 19184.553488426973,
                'ratio_upper_bound': 0.7149082531530696,
                'margin_lower_seconds': 5469.357866494018,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12471.486984250692,
                'mean_overwrite_seconds': 9183.09156867625,
                'ratio': 1.358092412667564,
                'margin_seconds': -3288.395415574443,
                'passed': False,
            },
            'confirmed_pass_count': 316_870,
            'unknown_pass_count': 2_083,
            'pass_lower': 0.5003347460035749,
            'pass_upper': 0.5036237833877559,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'log_l2',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 15606.838342836922,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.9472567973178312,
                'margin_lower_seconds': 868.9878397018238,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 13486.510556764477,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.573451148541981,
                'margin_seconds': -4915.21772110092,
                'passed': False,
            },
            'confirmed_pass_count': 345_873,
            'unknown_pass_count': 1_351,
            'pass_lower': 0.5056223631467691,
            'pass_upper': 0.5075973534253144,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'raw_l1',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 13695.24448086254,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.8312326392091284,
                'margin_lower_seconds': 2780.581701676205,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12794.480592839462,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.4927130408617013,
                'margin_seconds': -4223.187757175905,
                'passed': False,
            },
            'confirmed_pass_count': 313_744,
            'unknown_pass_count': 2_617,
            'pass_lower': 0.45865384896514017,
            'pass_upper': 0.46247957032631926,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'address_median',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 13606.783035254508,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.8258634732184247,
                'margin_lower_seconds': 2869.0431472842374,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 14067.09513829478,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.6411870890426485,
                'margin_seconds': -5495.802302631224,
                'passed': False,
            },
            'confirmed_pass_count': 309_845,
            'unknown_pass_count': 5_892,
            'pass_lower': 0.45295400655503804,
            'pass_upper': 0.46156736164104,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'previous_model',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 18270.751267289423,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 1.108942948588093,
                'margin_lower_seconds': -1794.9250847506773,
                'status': 'undetermined',
            },
            'paired': {
                'mae_seconds': 15703.963334962757,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.8321580695063273,
                'margin_seconds': -7132.6704992992,
                'passed': False,
            },
            'confirmed_pass_count': 343_712,
            'unknown_pass_count': 26,
            'pass_lower': 0.5024632558248325,
            'pass_upper': 0.5025012645200525,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'previous_address_median',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 13700.983252859925,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.8315809538814127,
                'margin_lower_seconds': 2774.84292967882,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12515.328889142658,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.4601448263520644,
                'margin_seconds': -3944.0360534791016,
                'passed': False,
            },
            'confirmed_pass_count': 312_892,
            'unknown_pass_count': 788,
            'pass_lower': 0.45740833326023966,
            'pass_upper': 0.4585602890999833,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'mix_log75',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 14651.078411138828,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.8892469639347249,
                'margin_lower_seconds': 1824.7477713999178,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12837.521943239293,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.4977346112624632,
                'margin_seconds': -4266.229107575737,
                'passed': False,
            },
            'confirmed_pass_count': 328_443,
            'unknown_pass_count': 1_746,
            'pass_lower': 0.4801419186204598,
            'pass_upper': 0.48269434869176997,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'mix_log50',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 14043.384235846148,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.8523629759295152,
                'margin_lower_seconds': 2432.4419466925974,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12545.000902254897,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.463606616035504,
                'margin_seconds': -3973.70806659134,
                'passed': False,
            },
            'confirmed_pass_count': 321_829,
            'unknown_pass_count': 1_955,
            'pass_lower': 0.4704730913056572,
            'pass_upper': 0.47333105281162013,
            'ninety_percent_status': 'failed',
        },
        {
            'workload': 'prn_1',
            'rule': 'range',
            'window': 'pooled',
            'method': 'mix_log25',
            'observed_reads': 684_054,
            'all_writes': 1_299_820,
            'paired_n': 591_037,
            'censored_n': 93_017,
            'observed_read': {
                'mae_seconds': 13741.737575063507,
                'mean_overwrite_lower_seconds': 16475.826182538745,
                'ratio_upper_bound': 0.8340545367993227,
                'margin_lower_seconds': 2734.088607475238,
                'status': 'passed',
            },
            'paired': {
                'mae_seconds': 12551.49233190359,
                'mean_overwrite_seconds': 8571.292835663557,
                'ratio': 1.4643639614877189,
                'margin_seconds': -3980.199496240033,
                'passed': False,
            },
            'confirmed_pass_count': 315_955,
            'unknown_pass_count': 2_169,
            'pass_lower': 0.46188604993173055,
            'pass_upper': 0.4650568522368117,
            'ninety_percent_status': 'failed',
        },
    ],
    'source_fingerprints': {
        'results/mean-overwrite/metrics.json': {
            'size_bytes': 95_792,
            'sha256': '28d7e5bb189327e653e6643b9ed524990e4abf9ad35e574405b2f16172e0c3b3',
        },
        'results/mean-overwrite/frozen.json': {
            'size_bytes': 992,
            'sha256': '86c338d46c9601214f2ead63c39f1683ace5062078c3ba38a8f379266c2d93e5',
        },
        'results/mean-overwrite/verification.json': {
            'size_bytes': 376,
            'sha256': '3dd2aa9fbbc2725c248945ddce4ddcd9a4c74a8484e50557da35c1c8516d79a0',
        },
        'results/long-history/proj_1/metrics.json': {
            'size_bytes': 132_353,
            'sha256': 'eb70165fb5cc2e7a700b450d0123d82a88df2b72a56e413c5f150e8eac2de0d5',
        },
        'results/long-history/proj_1/frozen.json': {
            'size_bytes': 34_817,
            'sha256': '50741a44557271512fb3e498deebec409ed9f0c827ca17bf6fe403c6cd69b3f5',
        },
        'results/long-history/proj_1/verification.json': {
            'size_bytes': 388,
            'sha256': '66a9004ed20adce1c452ba55293e97b32f215a35bc321b63fe4304028cfa7d34',
        },
        'results/long-history/prn_1/metrics.json': {
            'size_bytes': 134_067,
            'sha256': 'f952835efa9d0a2bb79756244fc3b53563dbd5c03838af0c33892fa7e1f6dfc3',
        },
        'results/long-history/prn_1/frozen.json': {
            'size_bytes': 35_083,
            'sha256': '05c3987d059ca6cdb356c96d9239e0494a5c44f83c7f8e22ca8c70d5ad458770',
        },
        'results/long-history/prn_1/verification.json': {
            'size_bytes': 388,
            'sha256': '369bdec39631a656ace9331ab801f2df843277cc0ee03021d93d25849ea14d55',
        },
        'results/mixture/prn_1/metrics.json': {
            'size_bytes': 133_493,
            'sha256': '6b3b3c3a0f6681b6a54ee0ca976dd320eef1b0583b9b6bb08488076407a7cade',
        },
        'results/mixture/prn_1/frozen.json': {
            'size_bytes': 34_689,
            'sha256': '45c75050dc953547bdd7c84113f47ef3680f8b68a5b7dd4ad8df94b0f1d0062d',
        },
        'results/mixture/prn_1/verification.json': {
            'size_bytes': 395,
            'sha256': 'c5a93ec95e3c540f7647d7e11d515bba973f50a1a9d4872912299bfa6feb9837',
        },
    },
}

# The embedded values make this notebook independent of the research repo.
for row in overwrite_results['selected_results']:
    metric = row['observed_read']
    n = row['observed_reads']
    assert abs(row['pass_lower'] - row['confirmed_pass_count'] / n) < 1e-10
    assert abs(row['pass_upper'] - (row['confirmed_pass_count'] + row['unknown_pass_count']) / n) < 1e-10
    if metric['status'] == 'passed':
        assert metric['mae_seconds'] < metric['mean_overwrite_lower_seconds']
    if row['ninety_percent_status'] == 'failed':
        assert row['pass_upper'] < 0.90
print('Embedded mean-overwrite and 90% results checked:', len(overwrite_results['selected_results']), 'rows')

Embedded mean-overwrite and 90% results checked: 20 rows


# Dataset selection: screening all 36 MSR Cambridge traces

Before training, every trace was screened with fixed windows and thresholds (train 0–14.4 h, validation 14.4–19.2 h from the trace start). A label counts only if the first next Read completes inside its own window; writes censored at the window end are not training examples. Future days were never opened during screening, so selection uses data availability, not later accuracy.

| Trace | Train labels | Validation labels | Validation addresses | Validation target Reads | Same-start eligible |
|---|---:|---:|---:|---:|---|
| prxy_1 (control, earlier stage) | 2,826,821 | 922,870 | 1,737 | 395,912 | yes |
| usr_1 (control, earlier stage) | 65,710 | 24,804 | 8,953 | 9,112 | yes |
| proj_1 | 182,142 | 21,556 | 16,515 | 16,888 | yes |
| prn_1 | 75,329 | 11,833 | 3,646 | 4,928 | yes |
| hm_0 | 22,502 | 5,032 | 76 | 2,632 | yes |
| prn_0 | 13,636 | 3,691 | 985 | 1,239 | yes |
| usr_0 | 11,599 | 2,963 | 1,013 | 1,059 | yes |
| proj_0 | 11,085 | 2,229 | 1,033 | 1,049 | yes |

The remaining 28 traces failed at least one threshold; the most common reasons are `validation_too_few_labels` (27), `train_too_few_labels` (25), `validation_too_few_labelled_offsets` (22), `train_too_few_labelled_offsets` (18). For example, `src1_0` and `src1_1` have many train labels but zero validation labels under this protocol.

## Range confirmation of the six new candidates

The six eligible new traces were re-labeled by byte-range intersection on the 512-byte grid and re-checked against the same thresholds.

| Trace | Train labels | Validation labels | Train target Reads | Validation coverage | Range eligible |
|---|---:|---:|---:|---:|---|
| proj_1 | 219,900 | 26,664 | 133,163 | 20.66% | yes |
| prn_1 | 82,136 | 16,739 | 61,508 | 21.46% | yes |
| hm_0 | 24,346 | 5,225 | 10,803 | 8.57% | yes |
| prn_0 | 32,249 | 5,629 | 5,107 | 3.66% | yes |
| usr_0 | 36,664 | 3,631 | 6,023 | 7.65% | yes |
| proj_0 | 19,484 | 2,345 | 3,818 | 3.48% | yes |

## Why proj_1 and prn_1 were chosen

- **Most training signal.** proj_1 has 182,142 train and 21,556 validation same-start labels; prn_1 has 75,329 and 11,833 — ranks 1 and 2 among the six new candidates.
- **Many distinct target events, also under range.** proj_1 reaches 133,163 distinct completing Reads in train (20,166 in validation) and prn_1 61,508 (5,886). Our per-record criteria (mean overwrite, ≥90% pass) need many independent events, not repeated writes to one address.
- **Most diverse addresses.** The largest single-address share of validation labels is 0.35% for proj_1 and 7.97% for prn_1, with effective address counts 8306 and 80.5. By comparison, hm_0 concentrates 20.33% on one address (effective count 11.7). A per-address median baseline and per-address history features need that diversity.
- **Both labeling rules supported.** same_start and range pass the thresholds, so the same workflow evaluates both rules on the same traces.
- **Full traces available for future days.** Both traces run the full 168 h (`end_of_trace`), so frozen evaluation on days 5–7 is possible without running out of trace. This is what our development results use.

## How this connects to the task results

The screening thresholds are a starting filter, not a guarantee of predictive quality. What the selected traces actually delivered in our evaluation:

| Trace / rule | Observed Read / all Write, days 5–7 | Coverage |
|---|---:|---:|
| proj_1 / same_start | 211,844 / 1,022,738 | 20.71% |
| proj_1 / range | 236,034 / 1,022,738 | 23.08% |
| prn_1 / same_start | 633,316 / 1,299,820 | 48.72% |
| prn_1 / range | 684,054 / 1,299,820 | 52.63% |

Roughly half of the writes have an observed Read within their day window, and the rest are censored. That is why MAE is reported only for observed-Read records and the ≥90% criterion is bounded rather than declared failed outright. Full details: `results.ipynb`, `reports/mean-overwrite.ru.md` and the protocols under `docs/protocols/`.


In [2]:
dataset_comparison = {
    'protocol': {
        'train_window_hours': 14.4,
        'validation_window_hours': 4.8,
        'thresholds': {
            'train_observed_labels': 10_000,
            'validation_observed_labels': 2_000,
            'train_distinct_offsets': 100,
            'validation_distinct_offsets': 50,
            'train_distinct_target_reads': 1_000,
            'validation_distinct_target_reads': 200,
        },
        'censoring': 'labels must complete inside their own window; future days not inspected',
    },
    'traces': [
        {
            'workload': 'prxy_1',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 2_826_821,
            'validation_observed_labels': 922_870,
            'train_coverage': 0.5409242679120513,
            'validation_coverage': 0.5356336515859427,
            'train_distinct_offsets': 1_806,
            'validation_distinct_offsets': 1_737,
            'train_distinct_target_reads': 1_213_227,
            'validation_distinct_target_reads': 395_912,
            'train_largest_address_share': 0.0007895795312119161,
            'validation_largest_address_share': 0.0008224343623695645,
            'train_effective_address_count': 1424.2934594319036,
            'validation_effective_address_count': 1353.0024790304712,
        },
        {
            'workload': 'usr_1',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 65_710,
            'validation_observed_labels': 24_804,
            'train_coverage': 0.1971733781431915,
            'validation_coverage': 0.15246736003540606,
            'train_distinct_offsets': 38_918,
            'validation_distinct_offsets': 8_953,
            'train_distinct_target_reads': 46_200,
            'validation_distinct_target_reads': 9_112,
            'train_largest_address_share': 0.0034241363567189165,
            'validation_largest_address_share': 0.004757297210127399,
            'train_effective_address_count': 9309.143338852668,
            'validation_effective_address_count': 1999.9688450836086,
        },
        {
            'workload': 'proj_1',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 182_142,
            'validation_observed_labels': 21_556,
            'train_coverage': 0.3178449462789655,
            'validation_coverage': 0.16703732690683384,
            'train_distinct_offsets': 95_839,
            'validation_distinct_offsets': 16_515,
            'train_distinct_target_reads': 105_229,
            'validation_distinct_target_reads': 16_888,
            'train_largest_address_share': 0.0012792217061413623,
            'validation_largest_address_share': 0.0035257005010205975,
            'train_effective_address_count': 5762.624901512025,
            'validation_effective_address_count': 8306.123056022308,
        },
        {
            'workload': 'usr_2',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': ['train_too_few_labels'],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 6_649,
            'validation_observed_labels': 18_471,
            'train_coverage': 0.045674364927803045,
            'validation_coverage': 0.30196174595389896,
            'train_distinct_offsets': 3_668,
            'validation_distinct_offsets': 2_598,
            'train_distinct_target_reads': 3_980,
            'validation_distinct_target_reads': 2_607,
            'train_largest_address_share': 0.009023913370431643,
            'validation_largest_address_share': 0.0033566130691354014,
            'train_effective_address_count': 950.5106533938208,
            'validation_effective_address_count': 1105.8138112539746,
        },
        {
            'workload': 'prn_1',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 75_329,
            'validation_observed_labels': 11_833,
            'train_coverage': 0.2752327433757655,
            'validation_coverage': 0.15170512820512821,
            'train_distinct_offsets': 36_411,
            'validation_distinct_offsets': 3_646,
            'train_distinct_target_reads': 59_280,
            'validation_distinct_target_reads': 4_928,
            'train_largest_address_share': 0.015850469274781294,
            'validation_largest_address_share': 0.07969238570100566,
            'train_effective_address_count': 2460.2787430265575,
            'validation_effective_address_count': 80.50215226429012,
        },
        {
            'workload': 'hm_0',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 22_502,
            'validation_observed_labels': 5_032,
            'train_coverage': 0.10844755460451487,
            'validation_coverage': 0.08255676597978738,
            'train_distinct_offsets': 744,
            'validation_distinct_offsets': 76,
            'train_distinct_target_reads': 10_490,
            'validation_distinct_target_reads': 2_632,
            'train_largest_address_share': 0.05337303350813261,
            'validation_largest_address_share': 0.20329888712241653,
            'train_effective_address_count': 57.99683820217821,
            'validation_effective_address_count': 11.691000694413663,
        },
        {
            'workload': 'prn_0',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 13_636,
            'validation_observed_labels': 3_691,
            'train_coverage': 0.03515339004898169,
            'validation_coverage': 0.023995267256959343,
            'train_distinct_offsets': 3_615,
            'validation_distinct_offsets': 985,
            'train_distinct_target_reads': 4_097,
            'validation_distinct_target_reads': 1_239,
            'train_largest_address_share': 0.1822381930184805,
            'validation_largest_address_share': 0.1761040368463831,
            'train_effective_address_count': 20.10111458512833,
            'validation_effective_address_count': 23.122036867000794,
        },
        {
            'workload': 'usr_0',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 11_599,
            'validation_observed_labels': 2_963,
            'train_coverage': 0.0898590021691974,
            'validation_coverage': 0.06238682780982861,
            'train_distinct_offsets': 2_604,
            'validation_distinct_offsets': 1_013,
            'train_distinct_target_reads': 3_425,
            'validation_distinct_target_reads': 1_059,
            'train_largest_address_share': 0.12432106216053108,
            'validation_largest_address_share': 0.028687141410732364,
            'train_effective_address_count': 26.82118373665413,
            'validation_effective_address_count': 144.66161907429682,
        },
        {
            'workload': 'proj_0',
            'full_window': True,
            'basic_eligible': True,
            'basic_failure_reasons': [],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 11_085,
            'validation_observed_labels': 2_229,
            'train_coverage': 0.08767281470467272,
            'validation_coverage': 0.03310412427784296,
            'train_distinct_offsets': 1_865,
            'validation_distinct_offsets': 1_033,
            'train_distinct_target_reads': 3_443,
            'validation_distinct_target_reads': 1_049,
            'train_largest_address_share': 0.13676138926477222,
            'validation_largest_address_share': 0.049349484073575596,
            'train_effective_address_count': 38.64072185064558,
            'validation_effective_address_count': 89.53920596875057,
        },
        {
            'workload': 'wdev_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 107,
            'validation_observed_labels': 1_726,
            'train_coverage': 0.0013534538371048736,
            'validation_coverage': 0.0551843207468747,
            'train_distinct_offsets': 25,
            'validation_distinct_offsets': 354,
            'train_distinct_target_reads': 77,
            'validation_distinct_target_reads': 706,
            'train_largest_address_share': 0.11214953271028037,
            'validation_largest_address_share': 0.028968713789107765,
            'train_effective_address_count': 15.164238410596026,
            'validation_effective_address_count': 132.9944642857143,
        },
        {
            'workload': 'src1_2',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': ['validation_too_few_labels'],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 78_829,
            'validation_observed_labels': 1_657,
            'train_coverage': 0.20276304485382252,
            'validation_coverage': 0.0617247159620041,
            'train_distinct_offsets': 8_468,
            'validation_distinct_offsets': 321,
            'train_distinct_target_reads': 10_449,
            'validation_distinct_target_reads': 1_065,
            'train_largest_address_share': 0.010364206066295399,
            'validation_largest_address_share': 0.03500301750150875,
            'train_effective_address_count': 714.1664134602312,
            'validation_effective_address_count': 140.39213580815053,
        },
        {
            'workload': 'web_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': ['train_too_few_labels', 'validation_too_few_labels'],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 8_831,
            'validation_observed_labels': 578,
            'train_coverage': 0.07324132897639624,
            'validation_coverage': 0.014457228614307153,
            'train_distinct_offsets': 1_648,
            'validation_distinct_offsets': 62,
            'train_distinct_target_reads': 2_106,
            'validation_distinct_target_reads': 209,
            'train_largest_address_share': 0.19793907824708415,
            'validation_largest_address_share': 0.42214532871972316,
            'train_effective_address_count': 16.424409319636386,
            'validation_effective_address_count': 4.026369706173018,
        },
        {
            'workload': 'proj_2',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': ['train_too_few_labels', 'validation_too_few_labels'],
            'target_event_support': True,
            'read_first_support': True,
            'train_observed_labels': 7_280,
            'validation_observed_labels': 414,
            'train_coverage': 0.1775696375432948,
            'validation_coverage': 0.00976968095148197,
            'train_distinct_offsets': 6_830,
            'validation_distinct_offsets': 414,
            'train_distinct_target_reads': 6_866,
            'validation_distinct_target_reads': 414,
            'train_largest_address_share': 0.0013736263736263737,
            'validation_largest_address_share': 0.0024154589371980675,
            'train_effective_address_count': 6144.029677718526,
            'validation_effective_address_count': 414.0,
        },
        {
            'workload': 'proj_3',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 36,
            'validation_observed_labels': 302,
            'train_coverage': 0.018301982714794104,
            'validation_coverage': 0.04319221967963387,
            'train_distinct_offsets': 18,
            'validation_distinct_offsets': 239,
            'train_distinct_target_reads': 20,
            'validation_distinct_target_reads': 241,
            'train_largest_address_share': 0.1111111111111111,
            'validation_largest_address_share': 0.0695364238410596,
            'train_effective_address_count': 14.727272727272727,
            'validation_effective_address_count': 81.87073608617594,
        },
        {
            'workload': 'rsrch_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 2,
            'validation_observed_labels': 49,
            'train_coverage': 1.7113031573543253e-05,
            'validation_coverage': 0.0011044991434496438,
            'train_distinct_offsets': 2,
            'validation_distinct_offsets': 15,
            'train_distinct_target_reads': 2,
            'validation_distinct_target_reads': 30,
            'train_largest_address_share': 0.5,
            'validation_largest_address_share': 0.22448979591836735,
            'train_effective_address_count': 2.0,
            'validation_effective_address_count': 8.667870036101084,
        },
        {
            'workload': 'mds_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 27,
            'validation_observed_labels': 37,
            'train_coverage': 0.000279367181599015,
            'validation_coverage': 0.0012372099244298803,
            'train_distinct_offsets': 12,
            'validation_distinct_offsets': 12,
            'train_distinct_target_reads': 21,
            'validation_distinct_target_reads': 27,
            'train_largest_address_share': 0.14814814814814814,
            'validation_largest_address_share': 0.2702702702702703,
            'train_effective_address_count': 9.986301369863014,
            'validation_effective_address_count': 5.7280334728033475,
        },
        {
            'workload': 'ts_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 95,
            'validation_observed_labels': 35,
            'train_coverage': 0.0009213550708473558,
            'validation_coverage': 0.000944592880468518,
            'train_distinct_offsets': 44,
            'validation_distinct_offsets': 10,
            'train_distinct_target_reads': 69,
            'validation_distinct_target_reads': 25,
            'train_largest_address_share': 0.10526315789473684,
            'validation_largest_address_share': 0.2857142857142857,
            'train_effective_address_count': 21.642685851318944,
            'validation_effective_address_count': 5.168776371308017,
        },
        {
            'workload': 'stg_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 72,
            'validation_observed_labels': 20,
            'train_coverage': 0.0005142342908566286,
            'validation_coverage': 0.0003819928567335791,
            'train_distinct_offsets': 25,
            'validation_distinct_offsets': 11,
            'train_distinct_target_reads': 50,
            'validation_distinct_target_reads': 11,
            'train_largest_address_share': 0.1527777777777778,
            'validation_largest_address_share': 0.2,
            'train_effective_address_count': 13.090909090909092,
            'validation_effective_address_count': 8.333333333333334,
        },
        {
            'workload': 'web_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 7,
            'train_coverage': 0.0,
            'validation_coverage': 0.003068829460762823,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 2,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 2,
            'train_largest_address_share': None,
            'validation_largest_address_share': 0.5714285714285714,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 1.96,
        },
        {
            'workload': 'hm_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 1_017,
            'validation_observed_labels': 1,
            'train_coverage': 0.609346914319952,
            'validation_coverage': 0.0022988505747126436,
            'train_distinct_offsets': 255,
            'validation_distinct_offsets': 1,
            'train_distinct_target_reads': 690,
            'validation_distinct_target_reads': 1,
            'train_largest_address_share': 0.021632251720747297,
            'validation_largest_address_share': 1.0,
            'train_effective_address_count': 110.27710843373494,
            'validation_effective_address_count': 1.0,
        },
        {
            'workload': 'mds_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 1,
            'train_coverage': 0.0,
            'validation_coverage': 0.0019157088122605363,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 1,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 1,
            'train_largest_address_share': None,
            'validation_largest_address_share': 1.0,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 1.0,
        },
        {
            'workload': 'src2_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 77,
            'validation_observed_labels': 1,
            'train_coverage': 0.0008085687283419091,
            'validation_coverage': 3.0181390154830533e-05,
            'train_distinct_offsets': 24,
            'validation_distinct_offsets': 1,
            'train_distinct_target_reads': 54,
            'validation_distinct_target_reads': 1,
            'train_largest_address_share': 0.15584415584415584,
            'validation_largest_address_share': 1.0,
            'train_effective_address_count': 13.69284064665127,
            'validation_effective_address_count': 1.0,
        },
        {
            'workload': 'proj_4',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 619,
            'validation_observed_labels': 0,
            'train_coverage': 0.06357194207661497,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 606,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 606,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': 0.006462035541195477,
            'validation_largest_address_share': None,
            'train_effective_address_count': 588.5729646697389,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'prxy_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 288,
            'validation_observed_labels': 0,
            'train_coverage': 0.0002754267680390188,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 101,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 143,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': 0.03819444444444445,
            'validation_largest_address_share': None,
            'train_effective_address_count': 60.191582002902756,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'rsrch_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 0,
            'train_coverage': 0.0,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': None,
            'validation_largest_address_share': None,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'rsrch_2',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 0,
            'train_coverage': 0.0,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': None,
            'validation_largest_address_share': None,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'src1_0',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': ['validation_too_few_labels', 'validation_too_few_labelled_offsets'],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 499_939,
            'validation_observed_labels': 0,
            'train_coverage': 0.21145891888165189,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 435_606,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 488_916,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': 0.00017802171864967527,
            'validation_largest_address_share': None,
            'train_effective_address_count': 335067.1755402936,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'src1_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': ['validation_too_few_labels', 'validation_too_few_labelled_offsets'],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 120_625,
            'validation_observed_labels': 0,
            'train_coverage': 0.5671293031303186,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 49_597,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 86_540,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': 0.0010943005181347151,
            'validation_largest_address_share': None,
            'train_effective_address_count': 28238.21668319512,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'src2_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 0,
            'train_coverage': None,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': None,
            'validation_largest_address_share': None,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'src2_2',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 0,
            'train_coverage': 0.0,
            'validation_coverage': None,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': None,
            'validation_largest_address_share': None,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'stg_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 52,
            'validation_observed_labels': 0,
            'train_coverage': 0.0009220185112947268,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 4,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 4,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': 0.25,
            'validation_largest_address_share': None,
            'train_effective_address_count': 4.0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'wdev_1',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 0,
            'train_coverage': 0.0,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': None,
            'validation_largest_address_share': None,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'wdev_2',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 0,
            'train_coverage': 0.0,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': None,
            'validation_largest_address_share': None,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'wdev_3',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 0,
            'validation_observed_labels': 0,
            'train_coverage': 0.0,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 0,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 0,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': None,
            'validation_largest_address_share': None,
            'train_effective_address_count': 0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'web_2',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'train_too_few_labelled_offsets',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 3,
            'validation_observed_labels': 0,
            'train_coverage': 0.0011636927851047323,
            'validation_coverage': 0.0,
            'train_distinct_offsets': 3,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 3,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': 0.3333333333333333,
            'validation_largest_address_share': None,
            'train_effective_address_count': 3.0,
            'validation_effective_address_count': 0,
        },
        {
            'workload': 'web_3',
            'full_window': True,
            'basic_eligible': False,
            'basic_failure_reasons': [
                'train_too_few_labels',
                'validation_too_few_labels',
                'validation_too_few_labelled_offsets',
            ],
            'target_event_support': False,
            'read_first_support': False,
            'train_observed_labels': 421,
            'validation_observed_labels': 0,
            'train_coverage': 0.11796021294480247,
            'validation_coverage': None,
            'train_distinct_offsets': 421,
            'validation_distinct_offsets': 0,
            'train_distinct_target_reads': 421,
            'validation_distinct_target_reads': 0,
            'train_largest_address_share': 0.0023752969121140144,
            'validation_largest_address_share': None,
            'train_effective_address_count': 421.0,
            'validation_effective_address_count': 0,
        },
    ],
    'eligible_new_workloads': ['proj_1', 'prn_1', 'hm_0', 'prn_0', 'usr_0', 'proj_0'],
    'validation_label_ranks': {
        'proj_1': 1,
        'prn_1': 2,
        'hm_0': 3,
        'prn_0': 4,
        'usr_0': 5,
        'proj_0': 6,
    },
    'range_confirmation': [
        {
            'workload': 'proj_1',
            'train_observed_labels': 219_900,
            'validation_observed_labels': 26_664,
            'train_coverage': 0.383734139774157,
            'validation_coverage': 0.206619191159947,
            'train_distinct_target_reads': 133_163,
            'validation_distinct_target_reads': 20_166,
            'basic_eligible': True,
        },
        {
            'workload': 'prn_1',
            'train_observed_labels': 82_136,
            'validation_observed_labels': 16_739,
            'train_coverage': 0.30010376627742136,
            'validation_coverage': 0.2146025641025641,
            'train_distinct_target_reads': 61_508,
            'validation_distinct_target_reads': 5_886,
            'basic_eligible': True,
        },
        {
            'workload': 'hm_0',
            'train_observed_labels': 24_346,
            'validation_observed_labels': 5_225,
            'train_coverage': 0.11733464422724732,
            'validation_coverage': 0.08572319201995013,
            'train_distinct_target_reads': 10_803,
            'validation_distinct_target_reads': 2_679,
            'basic_eligible': True,
        },
        {
            'workload': 'prn_0',
            'train_observed_labels': 32_249,
            'validation_observed_labels': 5_629,
            'train_coverage': 0.0831374065480794,
            'validation_coverage': 0.03659424529651155,
            'train_distinct_target_reads': 5_107,
            'validation_distinct_target_reads': 1_319,
            'basic_eligible': True,
        },
        {
            'workload': 'usr_0',
            'train_observed_labels': 36_664,
            'validation_observed_labels': 3_631,
            'train_coverage': 0.2840409048651999,
            'validation_coverage': 0.076451762327873,
            'train_distinct_target_reads': 6_023,
            'validation_distinct_target_reads': 1_268,
            'basic_eligible': True,
        },
        {
            'workload': 'proj_0',
            'train_observed_labels': 19_484,
            'validation_observed_labels': 2_345,
            'train_coverage': 0.15410167990129395,
            'validation_coverage': 0.034826905083688534,
            'train_distinct_target_reads': 3_818,
            'validation_distinct_target_reads': 1_075,
            'basic_eligible': True,
        },
    ],
    'controls': ['usr_1', 'prxy_1'],
    'selected': ['proj_1', 'prn_1'],
    'selection_trace_windows': {
        'proj_1': {
            'requested_hours': 168,
            'stop_reason': 'end_of_trace',
        },
        'prn_1': {
            'requested_hours': 168,
            'stop_reason': 'end_of_trace',
        },
    },
    'task_fit': {
        'proj_1': {
            'same_start': {
                'observed_reads': 211_844,
                'all_writes': 1_022_738,
                'coverage': 0.20713418294812552,
            },
            'range': {
                'observed_reads': 236_034,
                'all_writes': 1_022_738,
                'coverage': 0.23078637930731039,
            },
        },
        'prn_1': {
            'same_start': {
                'observed_reads': 633_316,
                'all_writes': 1_299_820,
                'coverage': 0.4872336169623486,
            },
            'range': {
                'observed_reads': 684_054,
                'all_writes': 1_299_820,
                'coverage': 0.526268252527273,
            },
        },
    },
    'failed_traces': 28,
    'failed_reason_counts': {
        'train_too_few_labels': 25,
        'train_too_few_labelled_offsets': 18,
        'validation_too_few_labels': 27,
        'validation_too_few_labelled_offsets': 22,
    },
    'source_fingerprints': {
        'results/data-sufficiency/summary.json': {
            'size_bytes': 64_061,
            'sha256': '5fff68be0aa4295192654b54d77c5f2c752e1d70ca8bb3de422782c0e1de1030',
        },
        'results/data-sufficiency-range/summary.json': {
            'size_bytes': 12_749,
            'sha256': 'bec9ddea89cd5e6e4376e639cb4f0f47a702a7a1c88eca8ad91cf60c8368d590',
        },
        'results/new-workloads/proj_1/source.json': {
            'size_bytes': 548,
            'sha256': '091a91cc292bf1806d4354d1d7fd7462a3347bea1416ba4ccd87b7c6aa6b6f14',
        },
        'results/new-workloads/prn_1/source.json': {
            'size_bytes': 545,
            'sha256': '7f725008e72ecc5c289949fdd61682913a696325827025a47746eb6670c44d18',
        },
    },
}

# The embedded screening snapshot is independent of the research repo.
assert len(dataset_comparison['traces']) == 36
assert dataset_comparison['failed_traces'] == 28
assert dataset_comparison['selected'] == ['proj_1', 'prn_1']
assert len(dataset_comparison['eligible_new_workloads']) == 6
assert all(row['basic_eligible'] for row in dataset_comparison['range_confirmation'])
assert all(entry['requested_hours'] == 168 for entry in dataset_comparison['selection_trace_windows'].values())
print('Dataset screening snapshot checked:', len(dataset_comparison['traces']), 'traces,',
      dataset_comparison['failed_traces'], 'rejected,', len(dataset_comparison['eligible_new_workloads']), 'new candidates')

Dataset screening snapshot checked: 36 traces, 28 rejected, 6 new candidates


## Validation and individual days: both criteria

These predictors are selected by validation MAE. Mean overwrite time is represented by its lower bound; an undetermined status does not mean failure. The error < overwrite fraction uses the same observed-Read records.

In [3]:
from IPython.display import Markdown, display

table = ['| Workload / rule | Window | MAE, s | Mean overwrite ≥, s | MAE < mean | error < overwrite | ≥90% |',
         '|---|---|---:|---:|---|---:|---|']
mean_status = {'passed': 'proven', 'failed': 'no', 'undetermined': 'undetermined'}
pass_status = {'passed': 'yes', 'failed': 'no', 'undetermined': 'undetermined'}
for row in overwrite_results['selected_results']:
    metric = row['observed_read']
    table.append(f"| {row['workload']} / {row['rule']} | {row['window']} | {metric['mae_seconds']:.2f} | {metric['mean_overwrite_lower_seconds']:.2f} | {mean_status[metric['status']]} | {row['pass_lower']:.2%}–{row['pass_upper']:.2%} | {pass_status[row['ninety_percent_status']]} |")
display(Markdown('\n'.join(table)))

| Workload / rule | Window | MAE, s | Mean overwrite ≥, s | MAE < mean | error < overwrite | ≥90% |
|---|---|---:|---:|---|---:|---|
| proj_1 / same_start | validation | 11082.82 | 19501.21 | proven | 64.42%–65.08% | no |
| proj_1 / same_start | day5 | 20505.72 | 26151.60 | proven | 38.14%–41.75% | no |
| proj_1 / same_start | day6 | 33227.53 | 16777.20 | undetermined | 25.00%–26.38% | no |
| proj_1 / same_start | day7 | 9415.10 | 18712.91 | proven | 68.52%–69.43% | no |
| proj_1 / same_start | pooled | 18162.30 | 18974.56 | proven | 51.35%–52.72% | no |
| proj_1 / range | validation | 11018.69 | 17527.78 | proven | 59.74%–60.33% | no |
| proj_1 / range | day5 | 18079.90 | 23862.02 | proven | 34.71%–37.57% | no |
| proj_1 / range | day6 | 29778.95 | 14228.30 | undetermined | 21.85%–23.34% | no |
| proj_1 / range | day7 | 9593.86 | 15579.72 | proven | 62.41%–63.37% | no |
| proj_1 / range | pooled | 17146.95 | 16275.23 | undetermined | 45.79%–47.17% | no |
| prn_1 / same_start | validation | 12644.23 | 16488.49 | proven | 44.08%–44.52% | no |
| prn_1 / same_start | day5 | 13362.36 | 20073.93 | proven | 50.08%–50.71% | no |
| prn_1 / same_start | day6 | 12529.23 | 17898.40 | proven | 50.65%–50.77% | no |
| prn_1 / same_start | day7 | 15458.86 | 19347.57 | proven | 49.32%–49.46% | no |
| prn_1 / same_start | pooled | 13715.20 | 19184.55 | proven | 50.03%–50.36% | no |
| prn_1 / range | validation | 11695.65 | 14279.25 | proven | 44.89%–45.22% | no |
| prn_1 / range | day5 | 13678.48 | 17293.43 | proven | 46.55%–47.09% | no |
| prn_1 / range | day6 | 12910.44 | 15303.39 | proven | 46.00%–46.17% | no |
| prn_1 / range | day7 | 14703.99 | 16645.43 | proven | 45.92%–46.10% | no |
| prn_1 / range | pooled | 13741.74 | 16475.83 | proven | 46.19%–46.51% | no |

## Selected predictors: prn_1 / proj_1

`predict.py` loads the packaged models and checks their fingerprints. prn_1 uses 25% log-L2 + 75% raw-L1 **after conversion to seconds**; proj_1 uses training-only per-address medians with a global-median fallback.

Input columns (raw, **do not apply log1p yourself**): offset bytes, size bytes, prior Read count, prior Write count, seconds since last Read, seconds since last Write, last completed Read-to-Read interval. Missing history is NaN. History is keyed by the same starting Offset for both rules, available before the current Write. The labeling rules differ, not the feature history. See models.json.

In [4]:
from pathlib import Path
import importlib.util
import numpy as np

model_directory = Path('submission') if Path('submission/models.json').is_file() else Path('.')
spec = importlib.util.spec_from_file_location('submission_predict', model_directory / 'predict.py')
predict_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(predict_module)
selected_predictor = predict_module.SelectedPredictor(model_directory)

# Illustrative input, not a measured evaluation sample.
raw_features = [[4096, 512, 0, 0, np.nan, np.nan, np.nan]]
for workload in ('prn_1', 'proj_1'):
    for rule in ('same_start', 'range'):
        prediction = selected_predictor.predict_seconds(workload, rule, raw_features)
        assert np.isfinite(prediction).all() and (prediction >= 0).all()
        print(workload, rule, 'predicted seconds:', prediction)

prn_1 same_start predicted seconds: [29029.04152598]
prn_1 range predicted seconds: [28107.32295629]
proj_1 same_start predicted seconds: [30913.136146]
proj_1 range predicted seconds: [27138.27158845]


## Earlier frozen-model experiment: usr_1 / prxy_1

The inference code and two model files below belong to the earlier days 2–4 experiment. The proj_1/prn_1 results above use different predictors and days 5–7; their recorded metrics are embedded for review.

**Features** — history of the same starting offset, available at the time of the write.

| Column | Feature before transformation |
|---|---|
| 0 | Write offset, bytes |
| 1 | Write size, bytes |
| 2 | Number of previous reads at this offset |
| 3 | Number of previous writes at this offset |
| 4 | Seconds since the previous read at this offset |
| 5 | Seconds since the previous write at this offset |
| 6 | Most recent completed interval between reads at this offset, seconds |

Each feature is transformed with `log1p`. Missing history is represented by `NaN`. usr_1 uses all seven columns; prxy_1 uses `[0, 1, 4, 5, 6]`.

**Label** — seconds from a write to the first subsequent read with the same starting offset. Intervening writes do not invalidate the sample. Samples without a read before the end of their time window are excluded from training/evaluation. The model is trained on `log1p(label)`; predictions in seconds are obtained as `max(0, expm1(prediction))`.


In [5]:
from pathlib import Path

import lightgbm as lgb
import numpy as np

model_directory = Path('.')
if Path('submission/usr_1.lightgbm.txt').is_file():
    model_directory = Path('submission')

models = {}
for workload in ('usr_1', 'prxy_1'):
    model_path = model_directory / f'{workload}.lightgbm.txt'
    models[workload] = lgb.Booster(model_file=str(model_path))

In [6]:
{workload: model.feature_name() for workload, model in models.items()}

{'usr_1': ['log_offset_bytes',
  'log_size_bytes',
  'log_prior_read_count',
  'log_prior_write_count',
  'log_seconds_since_read',
  'log_seconds_since_write',
  'log_last_read_read_interval_seconds'],
 'prxy_1': ['log_offset_bytes',
  'log_size_bytes',
  'log_seconds_since_read',
  'log_seconds_since_write',
  'log_last_read_read_interval_seconds']}

In [7]:
FEATURE_COLUMNS = {
    'usr_1': [0, 1, 2, 3, 4, 5, 6],
    'prxy_1': [0, 1, 4, 5, 6],
}


def predict_seconds(workload, raw_features):
    raw_features = np.atleast_2d(np.asarray(raw_features, dtype=float))
    log_features = np.log1p(raw_features)
    selected_features = log_features[:, FEATURE_COLUMNS[workload]]

    log_predictions = models[workload].predict(selected_features, num_threads=2)
    return np.maximum(0, np.expm1(log_predictions))

In [8]:
recorded_results = [
    {
        'workload': 'usr_1',
        'day': 2,
        'observed_intervals': 66_654,
        'coverage_fraction': 0.1783828484412175,
        'baseline_mae_seconds': 9918.773740623254,
        'model_mae_seconds': 12153.362087755688,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'usr_1',
        'day': 3,
        'observed_intervals': 3_577,
        'coverage_fraction': 0.02094102907858301,
        'baseline_mae_seconds': 13291.457813713294,
        'model_mae_seconds': 15071.470434888139,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'usr_1',
        'day': 4,
        'observed_intervals': 215_104,
        'coverage_fraction': 0.34996461372011484,
        'baseline_mae_seconds': 16450.66607042449,
        'model_mae_seconds': 24658.94281783792,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'prxy_1',
        'day': 2,
        'observed_intervals': 4_451_738,
        'coverage_fraction': 0.5392348543550198,
        'baseline_mae_seconds': 10.04810576824427,
        'model_mae_seconds': 5.559757821067748,
        'evaluation_role': 'previously_evaluated_replication',
    },
    {
        'workload': 'prxy_1',
        'day': 3,
        'observed_intervals': 4_313_417,
        'coverage_fraction': 0.5369413682602354,
        'baseline_mae_seconds': 11.015346296963763,
        'model_mae_seconds': 7.926027052344753,
        'evaluation_role': 'new_future_period',
    },
    {
        'workload': 'prxy_1',
        'day': 4,
        'observed_intervals': 4_270_777,
        'coverage_fraction': 0.5371556006332,
        'baseline_mae_seconds': 13.665613102591298,
        'model_mae_seconds': 9.647575508444456,
        'evaluation_role': 'new_future_period',
    },
]

In [9]:
header = (
    f'{"trace":<8} {"day":>3} {"n":>9} {"coverage_%":>11} '
    f'{"baseline_mae_s":>15} {"model_mae_s":>12}  {"evaluation_role"}'
)
print(header)

for result in recorded_results:
    coverage_percent = 100 * result['coverage_fraction']
    print(
        f'{result["workload"]:<8} {result["day"]:>3} '
        f'{result["observed_intervals"]:>9,} {coverage_percent:>11.2f} '
        f'{result["baseline_mae_seconds"]:>15.2f} '
        f'{result["model_mae_seconds"]:>12.2f}  '
        f'{result["evaluation_role"]}'
    )

trace    day         n  coverage_%  baseline_mae_s  model_mae_s  evaluation_role
usr_1      2    66,654       17.84         9918.77     12153.36  new_future_period
usr_1      3     3,577        2.09        13291.46     15071.47  new_future_period
usr_1      4   215,104       35.00        16450.67     24658.94  new_future_period
prxy_1     2 4,451,738       53.92           10.05         5.56  previously_evaluated_replication
prxy_1     3 4,313,417       53.69           11.02         7.93  new_future_period
prxy_1     4 4,270,777       53.72           13.67         9.65  new_future_period


*Note: MAE is the average absolute prediction error in seconds; lower is better. On days 2–4, the prxy_1 model reduces MAE by 28–45% compared with the training-only per-address median, while the usr_1 model performs worse on all three days. Thus, the benefit is workload-dependent. Scores include only writes followed by a read within the same day; coverage indicates the fraction evaluated. Day 2 of prxy_1 repeats a previously evaluated period.*